# OpenVLA 复现课程笔记

这份 notebook 按照从硬件检测到最终验证的顺序，记录 OpenVLA-7B + QLoRA 的复现过程。

## 1. 硬件设备检测

先确认操作系统、Python、GPU、显存和 CUDA 是否正常。

In [9]:
import os
import platform
import subprocess
import torch

print('操作系统:', platform.system(), platform.release())
print('Python 版本:', platform.python_version())
print('Conda 环境:', os.environ.get('CONDA_DEFAULT_ENV', 'not-set'))
print('CUDA 可用:', torch.cuda.is_available())
print('CUDA 版本:', torch.version.cuda)
print('GPU 数量:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('当前 GPU:', torch.cuda.get_device_name(0))
    print('GPU 显存:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,memory.free', '--format=csv,noheader'], text=True))
except Exception as error:
    print('nvidia-smi 不可用:', error)

操作系统: Linux 6.8.0-138-generic
Python 版本: 3.10.20
Conda 环境: openvla
CUDA 可用: True
CUDA 版本: 12.8
GPU 数量: 1
当前 GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
GPU 显存: 11.47 GB
NVIDIA GeForce RTX 5070 Ti Laptop GPU, 595.91.07, 12227 MiB, 3100 MiB



## 2. Python 环境与依赖检查

确认 OpenVLA 所需的主要依赖可以正常导入。

In [10]:
import importlib

required_packages = ['torch', 'transformers', 'peft', 'bitsandbytes', 'PIL', 'numpy']
for package_name in required_packages:
    try:
        module = importlib.import_module(package_name)
        print(f'{package_name}: OK -> {getattr(module, "__version__", "unknown")}')
    except Exception as error:
        print(f'{package_name}: MISSING -> {error}')

torch: OK -> 2.8.0+cu128
transformers: OK -> 4.40.1
peft: OK -> 0.11.1
bitsandbytes: OK -> 0.50.2
PIL: OK -> 12.3.0
numpy: OK -> 2.2.6


## 3. 仓库状态与关键文件

确认复现入口、训练脚本、adapter 和文档目录都存在。

In [ ]:
import os
from pathlib import Path

repo = Path(os.environ.get('OPENVLA_REPO', Path.cwd())).expanduser().resolve()
if not (repo / 'reproduction' / 'test_planB_libero_compare.py').exists():
    raise FileNotFoundError(
        f'当前目录不是 OpenVLA 仓库，请设置 OPENVLA_REPO。当前路径: {repo}'
    )

key_paths = [
    'reproduction/run_qlora_1000.py',
    'vla-scripts/finetune.py',
    'reproduction/test_1000step_adapter.py',
    'reproduction/test_planB_libero_compare.py',
    'reproduction/test_planB2_libero_gt.py',
    'adapter-tmp',
    'AIGC_document',
]
for relative_path in key_paths:
    path = repo / relative_path
    print(relative_path, '-> exists' if path.exists() else '-> missing')

run_qlora_1000.py -> exists
vla-scripts/finetune.py -> exists
test_1000step_adapter.py -> exists
test_planB_libero_compare.py -> exists
adapter-tmp -> exists
AIGC_document -> exists


## 4. 单机兼容性修复

主要修改集中在训练入口，而不是 OpenVLA 基础模型结构。单卡运行时只在分布式进程组已初始化时使用 DDP 和 barrier；保存模型时同时兼容 DDP 包装对象和普通模型对象。

In [12]:
def unwrap_model_for_saving(model):
    return model.module if hasattr(model, 'module') else model

print('单机模式：不包装 DDP，不调用 dist.barrier()')
print('分布式模式：使用 DDP，并通过 model.module 保存原始模型')

单机模式：不包装 DDP，不调用 dist.barrier()
分布式模式：使用 DDP，并通过 model.module 保存原始模型


## 5. 本地缓存与离线运行

使用本地 Hugging Face 缓存，避免 Hub 网络访问影响复现。

In [13]:
import os
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

hf_home = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface'))
base_cache = hf_home / 'hub' / 'models--openvla--openvla-7b' / 'snapshots'
adapter_dir = repo / 'adapter-tmp' / 'libero_spatial_1000step'
print('仓库路径:', repo)
print('base 模型缓存存在:', base_cache.exists())
print('adapter 目录存在:', adapter_dir.exists())
print('HF_HUB_OFFLINE =', os.environ['HF_HUB_OFFLINE'])
print('TRANSFORMERS_OFFLINE =', os.environ['TRANSFORMERS_OFFLINE'])

仓库路径: /home/yyz/openvla
base 模型缓存存在: True
adapter 目录存在: True
HF_HUB_OFFLINE = 1
TRANSFORMERS_OFFLINE = 1


## 6. 显存检查

加载 4-bit 模型前，先确认没有其他 Python、Jupyter 或实验进程占用大量显存。

In [14]:
import subprocess

try:
    output = subprocess.check_output(
        ['nvidia-smi', '--query-compute-apps=pid,process_name,used_memory', '--format=csv,noheader'],
        text=True,
    )
    print(output or '当前没有计算进程')
except Exception as error:
    print('无法查看 GPU 进程:', error)

105336, /home/yyz/miniconda3/envs/fast3r/bin/python, 7202 MiB



## 7. adapter smoke test

执行 1000-step adapter 的最小推理验证，确认 4-bit base 模型、LoRA adapter 和推理流程都能工作。

In [ ]:
import os
import subprocess
import sys

environment = os.environ.copy()
environment['HF_HUB_OFFLINE'] = '1'
environment['TRANSFORMERS_OFFLINE'] = '1'
environment['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.run(
    [sys.executable, str(repo / 'reproduction' / 'test_1000step_adapter.py')],
    cwd=repo,
    env=environment,
    text=True,
    capture_output=True,
    check=False,
)
print('returncode:', result.returncode)
print(result.stdout)
if result.stderr:
    print(result.stderr)

returncode: 1
OpenVLA 1000-step QLoRA adapter smoke test

[1/5] Loading processor...
Processor loaded.

[2/5] Loading 4-bit base OpenVLA...

I0000 00:00:1789835609.825666  128530 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789835609.851930  128530 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789835610.635710  128530 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environme

## 8. LIBERO 对比验证

运行 base model 与 adapter 的对比脚本，确认 adapter 不仅能加载，而且会改变动作输出。

In [ ]:
import os
import subprocess
import sys

environment = os.environ.copy()
environment['HF_HUB_OFFLINE'] = '1'
environment['TRANSFORMERS_OFFLINE'] = '1'
environment['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.run(
    [sys.executable, str(repo / 'reproduction' / 'test_planB_libero_compare.py')],
    cwd=repo,
    env=environment,
    text=True,
    capture_output=True,
    check=False,
)
print('returncode:', result.returncode)
print('CSV exists:', (repo / 'planB_libero_compare.csv').exists())
print(result.stdout[-4000:])
if result.stderr:
    print(result.stderr[-2000:])

returncode: 1
CSV exists: True
OpenVLA Plan B
Base OpenVLA vs 1000-step LIBERO Spatial LoRA
Model      : openvla/openvla-7b
Image      : /home/yyz/openvla/test_image.jpg
Adapter    : /home/yyz/openvla/adapter-tmp/libero_spatial_1000step/openvla-7b+libero_spatial_no_noops+b16+lr-0.0005+lora-r32+dropout-0.0+q-4bit--image_aug
Statistics : /home/yyz/openvla/runs/libero_spatial_1000step/openvla-7b+libero_spatial_no_noops+b16+lr-0.0005+lora-r32+dropout-0.0+q-4bit--image_aug/dataset_statistics.json
Output     : /home/yyz/openvla/planB_libero_compare.csv
[1/7] Loading LIBERO statistics...
Dataset key: libero_spatial_no_noops
q01 : [-0.7454732  -0.66160715 -0.9375     -0.10714286 -0.20678571 -0.18428572
  0.        ]
q99 : [0.9375     0.8758929  0.93214285 0.10392857 0.17678571 0.14571428
 1.        ]
mask: [ True  True  True  True  True  True False]

[2/7] Loading image...
Image size: (686, 732)

[3/7] Loading processor...
Processor loaded.

[4/7] Using OpenVLA action decoding...
Action decodi

## 9. 复现结论

本地复现依赖本地模型缓存、离线环境和可用显存。单机兼容性修改位于训练入口，不是基础模型结构修改。验证 adapter 效果时不需要强制 merge。